# Application of Python to Estimate Asset Prices and value


In [1]:
# Standard Data Science Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import pyarrow

# Tiingo API
import tiingo
from tiingo import TiingoClient

# Web Requests and Querying
import requests
import  json
import os
#import pandas_datareader as web
#from aquarel import load_theme

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

#--- Set up Tiingo API client

config = {
    'session': True,
    'api_key': os.getenv("TIINGO_API_KEY")
}

client = TiingoClient(config)

In [2]:
# --- Config ---

TICKERS = ['SPY', 'KO', 'JNJ', 'NVDA', 'BRK-B', 'TFC', 'AAPL']
START_DATE = '2015-01-01'
END_DATE = '2025-12-31'
DATA_DIR = 'data'

os.makedirs(DATA_DIR, exist_ok=True)

In [3]:
# Define a Function to Fetch Daily Prices with Caching

def fetch_daily_prices(ticker, start=START_DATE, end=END_DATE, refresh=False):
    """
    Pull daily OHLCV + adjusted close from Tiingo, with local parquet caching.
    Set refresh=True to force a re-pull.
    """
    path = os.path.join(DATA_DIR, f'{ticker}_daily.parquet')

    if os.path.exists(path) and not refresh:
        return pd.read_parquet(path)

    df = client.get_dataframe(
        ticker,
        frequency='daily',
        startDate=start,
        endDate=end,
    )
    # Tiingo returns date as the index already with get_dataframe
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = 'date'

    df.to_parquet(path)
    return df

In [4]:
prices = {ticker: fetch_daily_prices(ticker) for ticker in TICKERS}

for ticker, df in prices.items():
    print(f"{ticker}: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

SPY: 2766 rows, 2015-01-02 to 2025-12-31
KO: 2766 rows, 2015-01-02 to 2025-12-31
JNJ: 2766 rows, 2015-01-02 to 2025-12-31
NVDA: 2766 rows, 2015-01-02 to 2025-12-31
BRK-B: 2766 rows, 2015-01-02 to 2025-12-31
TFC: 2766 rows, 2015-01-02 to 2025-12-31
AAPL: 2766 rows, 2015-01-02 to 2025-12-31


In [5]:
ohlcv = pd.concat(prices, axis=1)
# columns become a MultiIndex: (ticker, field) like ('SPY', 'adjClose')
print(ohlcv.head())
print(ohlcv.columns[:8])

               SPY                                                             \
             close    high     low    open     volume    adjClose     adjHigh   
date                                                                            
2015-01-02  205.43  206.88  204.18  206.38  121465865  169.972665  171.172394   
2015-01-05  201.72  204.37  201.35  204.17  169632646  166.903013  169.095622   
2015-01-06  199.82  202.72  198.86  202.09  209151408  165.330954  167.730413   
2015-01-07  202.31  202.72  200.88  201.42  125346709  167.391179  167.730413   
2015-01-08  205.90  206.16  203.99  204.01  147217784  170.361543  170.576667   

                                               ...     AAPL                    \
                adjLow     adjOpen  adjVolume  ...      low    open    volume   
date                                           ...                              
2015-01-02  168.938416  170.758695  121465865  ...  107.350  111.39  53204626   
2015-01-05  166.596875  168

In [6]:
french = pd.read_csv('/mnt/hard_storage/GithubRepos/project-dev/ssm-finance/data/F-F_Research_Data_5_Factors_2x3_daily.csv', skiprows=3)

In [7]:
french = french.rename(columns={'Unnamed: 0': 'date'})


# Drop the copyright footer and any other non-date rows
french = french[french['date'].astype(str).str.match(r'^\d{8}$')].copy()

french['date'] = pd.to_datetime(french['date'], format='%Y%m%d')
french = french.set_index('date')


french = french.astype(float) / 100

In [8]:
ohlcv_flat = ohlcv.copy()
ohlcv_flat.columns = [f'{ticker}_{field}' for ticker, field in ohlcv_flat.columns]

capm = ohlcv_flat.join(french, how='inner')
print(capm.head())
print(capm.columns.tolist()[:5], '...', capm.columns.tolist()[-7:])

            SPY_close  SPY_high  SPY_low  SPY_open  SPY_volume  SPY_adjClose  \
date                                                                           
2015-01-02     205.43    206.88   204.18    206.38   121465865    169.972665   
2015-01-05     201.72    204.37   201.35    204.17   169632646    166.903013   
2015-01-06     199.82    202.72   198.86    202.09   209151408    165.330954   
2015-01-07     202.31    202.72   200.88    201.42   125346709    167.391179   
2015-01-08     205.90    206.16   203.99    204.01   147217784    170.361543   

            SPY_adjHigh  SPY_adjLow  SPY_adjOpen  SPY_adjVolume  ...  \
date                                                             ...   
2015-01-02   171.172394  168.938416   170.758695      121465865  ...   
2015-01-05   169.095622  166.596875   168.930142      169632646  ...   
2015-01-06   167.730413  164.536651   167.209151      209151408  ...   
2015-01-07   167.730413  166.207998   166.654793      125346709  ...   
2015-01

In [9]:
capm.describe()

,SPY_close,SPY_high,SPY_low,SPY_open,SPY_volume,SPY_adjClose,SPY_adjHigh,SPY_adjLow,SPY_adjOpen,SPY_adjVolume,...,AAPL_adjOpen,AAPL_adjVolume,AAPL_divCash,AAPL_splitFactor,Mkt-RF,SMB,HML,RMW,CMA,RF
count,2766.000000,2766.000000,2766.000000,2766.000000,2.766000e+03,2766.000000,2766.000000,2766.000000,2766.000000,2.766000e+03,...,2766.000000,2.766000e+03,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000
mean,360.790821,362.678792,358.599151,360.732569,8.601431e+07,336.915235,338.681947,334.863682,336.860063,8.601431e+07,...,106.134319,1.111398e+08,0.007209,1.001085,0.000489,-0.000100,-0.000064,0.000113,-0.000060,0.000079
std,132.150089,132.750812,131.417801,132.136027,4.383520e+07,140.404033,141.051667,139.617594,140.386583,4.383520e+07,...,74.177626,6.793139e+07,0.063264,0.057042,0.011562,0.006851,0.008874,0.005294,0.004826,0.000085
min,182.860001,184.100006,181.020004,182.339996,2.027001e+07,154.447822,155.495159,152.325963,154.008614,2.027001e+07,...,20.506307,1.791057e+07,0.000000,1.000000,-0.120100,-0.045800,-0.050300,-0.022300,-0.029200,0.000000
25%,248.080000,249.777500,247.010000,247.965000,5.869342e+07,218.763012,219.167740,217.338395,218.449656,5.869342e+07,...,36.630409,6.473323e+07,0.000000,1.000000,-0.004000,-0.004100,-0.004675,-0.002900,-0.002700,0.000000
50%,326.655000,327.935000,323.770000,326.460000,7.554834e+07,299.764591,301.611058,298.085340,299.347997,7.554834e+07,...,88.365169,9.426003e+07,0.000000,1.000000,0.000600,-0.000300,-0.000400,0.000000,-0.000200,0.000100
75%,443.565000,445.505000,441.677500,443.702500,9.941888e+07,421.937570,424.104301,419.786375,422.252386,9.941888e+07,...,168.261941,1.369125e+08,0.000000,1.000000,0.006000,0.003600,0.004300,0.003000,0.002300,0.000200
max,690.380000,691.660000,689.270000,690.640000,5.072443e+08,688.472443,689.748906,687.365510,688.731725,5.072443e+08,...,285.929293,6.488252e+08,0.820000,4.000000,0.096500,0.057100,0.067300,0.042500,0.024700,0.000200
